In [1]:
# ============================================================
# Install Required Libraries
# ============================================================

!pip install -q torch torchvision torchaudio

!pip install -q transformers
!pip install -q accelerate
!pip install -q peft
!pip install -q bitsandbytes
!pip install -q datasets

!pip install -q sentencepiece
!pip install -q protobuf

!pip install -q qwen-vl-utils

print("=" * 60)
print("All Required Libraries Installed Successfully")
print("=" * 60)

All Required Libraries Installed Successfully


In [2]:
# ============================================================
# Import Required Libraries
# ============================================================

import torch

from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)

print("=" * 60)
print("Libraries Imported Successfully")
print("=" * 60)

Libraries Imported Successfully


In [3]:
# ============================================================
# Check GPU
# ============================================================

print("=" * 60)

print("CUDA Available :", torch.cuda.is_available())

if torch.cuda.is_available():

    print("GPU Name :", torch.cuda.get_device_name(0))
    print("GPU Count :", torch.cuda.device_count())

    device = torch.device("cuda")

else:

    print("Running on CPU")
    device = torch.device("cpu")

print("Device :", device)

print("=" * 60)

CUDA Available : True
GPU Name : NVIDIA GeForce RTX 3050 6GB Laptop GPU
GPU Count : 1
Device : cuda


In [4]:
# ============================================================
# BitsAndBytes Configuration
# ============================================================

bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_compute_dtype=torch.float16,

    bnb_4bit_use_double_quant=True

)

print("=" * 60)
print("4-bit Quantization Configured")
print("=" * 60)

4-bit Quantization Configured


In [5]:
# ============================================================
# Load Qwen2.5-VL Processor
# ============================================================

MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"

processor = AutoProcessor.from_pretrained(
    MODEL_NAME
)

print("=" * 60)
print("Processor Loaded Successfully")
print("=" * 60)

Processor Loaded Successfully


In [6]:
# ============================================================
# Load Student Model
# ============================================================

student_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(

    MODEL_NAME,

    quantization_config=bnb_config,

    torch_dtype=torch.float16,

    device_map="auto"

)

print("=" * 60)
print("Student Model Loaded Successfully")
print("=" * 60)

W0728 12:30:35.203000 11960 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Student Model Loaded Successfully


In [7]:
# ============================================================
# LoRA Configuration
# ============================================================

lora_config = LoraConfig(

    r=16,

    lora_alpha=32,

    lora_dropout=0.05,

    bias="none",

    task_type=TaskType.CAUSAL_LM,

    target_modules=[

        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",

        "gate_proj",
        "up_proj",
        "down_proj"

    ]

)

student_model = get_peft_model(

    student_model,

    lora_config

)

print("=" * 60)
print("LoRA Applied Successfully")
print("=" * 60)

LoRA Applied Successfully


In [8]:
# ============================================================
# Print Trainable Parameters
# ============================================================

student_model.print_trainable_parameters()

print("=" * 60)
print("LoRA Configuration Summary")
print("=" * 60)

trainable_params = 0
all_params = 0

for _, param in student_model.named_parameters():

    all_params += param.numel()

    if param.requires_grad:
        trainable_params += param.numel()

print(f"Total Parameters      : {all_params:,}")
print(f"Trainable Parameters  : {trainable_params:,}")
print(f"Frozen Parameters     : {all_params - trainable_params:,}")
print(f"Trainable Percentage  : {(100 * trainable_params / all_params):.4f}%")

print("=" * 60)

trainable params: 37,152,768 || all params: 3,791,775,744 || trainable%: 0.9798
LoRA Configuration Summary
Total Parameters      : 2,071,177,216
Trainable Parameters  : 37,152,768
Frozen Parameters     : 2,034,024,448
Trainable Percentage  : 1.7938%


In [9]:
# ============================================================
# Verify Student Model
# ============================================================

print("=" * 60)
print("Student Model Verification")
print("=" * 60)

print(f"Model Name : {MODEL_NAME}")

print(f"\nModel Device : {next(student_model.parameters()).device}")

print(f"\nProcessor Type : {type(processor).__name__}")

print(f"\nLoRA Applied : {'Yes' if hasattr(student_model, 'peft_config') else 'No'}")

print("\nModel Architecture:\n")
print(student_model.__class__.__name__)

print("=" * 60)
print("Student Model Ready for Fine-Tuning")
print("=" * 60)

Student Model Verification
Model Name : Qwen/Qwen2.5-VL-3B-Instruct

Model Device : cuda:0

Processor Type : Qwen2_5_VLProcessor

LoRA Applied : Yes

Model Architecture:

PeftModelForCausalLM
Student Model Ready for Fine-Tuning
